# Fresh-start ingest

One-time clean rebuild of the corpus:
1. Wipe `chroma_db/` and `data/processed/*.json` (keep eval data + source PDFs)
2. Re-parse PDFs through Docling
3. Re-embed all chunks into Chroma with stable content-hash IDs

After this completes, the corpus is fully reproducible — same content always
produces the same chunk_ids, so re-parsing is idempotent.

⚠️ **Cell 2 won't actually delete anything until you set `CONFIRM_DELETE = True`.**

In [2]:
"""Wipe chroma_db/ and data/processed/*.json. Eval data is preserved.

Default mode = DRY RUN: shows what will be deleted without doing it.
Set CONFIRM_DELETE = True, re-run the cell to actually delete.
"""
import shutil
from rag_pipeline.config import cfg, log

CONFIRM_DELETE = True   # ← flip to True, then re-run, to actually delete

to_delete = [
    cfg.CHROMA_PERSIST_DIR,
    cfg.DATA_PROCESSED_DIR / "phase0_chunks.json",
    cfg.DATA_PROCESSED_DIR / "phase1_chunks.json",
    cfg.DATA_PROCESSED_DIR / "ragas_rows_mq.json",
    cfg.DATA_PROCESSED_DIR / "ragas_quickstart_cache.json",
]
to_keep = [
    cfg.EVAL_SET_PATH,
    cfg.PROJECT_ROOT / "src" / "rag_pipeline" / "eval" / "data",
    cfg.DATA_RAW_DIR,
    cfg.EVAL_RESULTS_DIR,
]

print("=== WILL DELETE ===")
for p in to_delete:
    if p.exists():
        kind = "DIR " if p.is_dir() else "FILE"
        print(f"  {kind}  {p}")
    else:
        print(f"  (already gone)  {p}")

print("\n=== WILL KEEP ===")
for p in to_keep:
    mark = "✓" if p.exists() else "?"
    print(f"  {mark}  {p}")

if not CONFIRM_DELETE:
    print("\n⚠️  CONFIRM_DELETE=False — nothing deleted. Set True + re-run to wipe.")
else:
    print("\n🔥 Deleting...")
    for p in to_delete:
        if p.is_dir():
            shutil.rmtree(p, ignore_errors=True)
            log.info(f"  rm -rf {p}")
        elif p.is_file():
            p.unlink()
            log.info(f"  rm {p}")
    print("\n✅ Clean slate ready")

2026-06-03 23:29:46,452 - INFO    | rag -   rm -rf /home/thimu/github_vs/protoRAG/rag-pipeline/chroma_db
2026-06-03 23:29:46,453 - INFO    | rag -   rm /home/thimu/github_vs/protoRAG/rag-pipeline/data/processed/phase0_chunks.json
2026-06-03 23:29:46,454 - INFO    | rag -   rm /home/thimu/github_vs/protoRAG/rag-pipeline/data/processed/phase1_chunks.json
2026-06-03 23:29:46,454 - INFO    | rag -   rm /home/thimu/github_vs/protoRAG/rag-pipeline/data/processed/ragas_rows_mq.json


=== WILL DELETE ===
  DIR   /home/thimu/github_vs/protoRAG/rag-pipeline/chroma_db
  FILE  /home/thimu/github_vs/protoRAG/rag-pipeline/data/processed/phase0_chunks.json
  FILE  /home/thimu/github_vs/protoRAG/rag-pipeline/data/processed/phase1_chunks.json
  FILE  /home/thimu/github_vs/protoRAG/rag-pipeline/data/processed/ragas_rows_mq.json
  (already gone)  /home/thimu/github_vs/protoRAG/rag-pipeline/data/processed/ragas_quickstart_cache.json

=== WILL KEEP ===
  ✓  /home/thimu/github_vs/protoRAG/rag-pipeline/eval/eval_set.json
  ✓  /home/thimu/github_vs/protoRAG/rag-pipeline/src/rag_pipeline/eval/data
  ✓  /home/thimu/github_vs/protoRAG/rag-pipeline/data/raw
  ✓  /home/thimu/github_vs/protoRAG/rag-pipeline/eval/results

🔥 Deleting...

✅ Clean slate ready


In [ ]:
"""

Intial Config, LLM, Embedding Model, Data Corpus, Paths ::::::::::::::::::::::::


Verify config + define shared constants for the rest of the notebook."""

from pathlib import Path
from rag_pipeline.config import cfg, log

log.info(f"Provider:    {cfg.MODEL_PROVIDER}")
log.info(f"LLM:         {cfg.OLLAMA_MODEL}")
log.info(f"Embeddings:  {cfg.OLLAMA_EMBEDDING_MODEL}")

# Where your IPC PDFs live (NOT under data/raw/ — they're in your Downloads folder)
PDF_SOURCE_DIR = Path("/home/thimu/Downloads/pdf_splitter/IPC")
assert PDF_SOURCE_DIR.exists(), f"PDFs not found at {PDF_SOURCE_DIR}"

CHUNKS_CACHE = cfg.DATA_PROCESSED_DIR / "phase1_chunks.json"
COLLECTION   = "IPC_Corpus"

log.info(f"PDF source:  {PDF_SOURCE_DIR}")
log.info(f"PDFs found:  {len(list(PDF_SOURCE_DIR.glob('*.pdf')))}")

2026-06-03 23:32:04,305 - INFO    | rag - Provider:    ollama
2026-06-03 23:32:04,306 - INFO    | rag - LLM:         gemma-4-e4b:latest
2026-06-03 23:32:04,307 - INFO    | rag - Embeddings:  embeddinggemma:latest
2026-06-03 23:32:04,378 - INFO    | rag - PDF source:  /home/thimu/Downloads/pdf_splitter/IPC
2026-06-03 23:32:04,904 - INFO    | rag - PDFs found:  74


In [ ]:
"""

Parsing:::::::::::::::::::::

Parse every PDF under PDF_SOURCE_DIR with the production dispatcher.

Wall time: ~15-18 min for 63 IPC PDFs on CPU (Docling falls back from CUDA).
The resulting chunks are saved to phase1_chunks.json so this never has to
run again unless the source PDFs change.
"""
from rag_pipeline.parsers import default_dispatcher, save_chunks_cache

dispatcher = default_dispatcher()
chunks = dispatcher.parse_directory(PDF_SOURCE_DIR)

assert chunks, "Parsing produced 0 chunks — check PDF_SOURCE_DIR"

save_chunks_cache(chunks, CHUNKS_CACHE)
log.info(f"✅ Parsed and cached {len(chunks)} chunks")

/home/thimu/github_vs/protoRAG/rag-pipeline/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
2026-06-03 23:33:04,088 - INFO    | rag - Found 74 supported files under /home/thimu/Downloads/pdf_splitter/IPC
Parsing:   0%|          | 0/74 [00:00<?, ?file/s]/home/thimu/github_vs/protoRAG/rag-pipeline/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version t

In [ ]:
"""

Embedding:::::::::::::::::::


Create an empty Chroma collection and embed every chunk.

Because Cell 2 wiped chroma_db/, the collection starts empty — no need
for the idempotent dedup logic here. Every chunk gets embedded once.
"""
from rag_pipeline.vectorstore import get_vectorstore
from tqdm import tqdm

vs = get_vectorstore(COLLECTION)
assert vs._collection.count() == 0, "Collection not empty — did Cell 2 actually wipe?"

docs = [c.to_langchain_document() for c in chunks]
ids  = [c.chunk_id for c in chunks]

BATCH = 64
for i in tqdm(range(0, len(chunks), BATCH), desc="Embedding"):
    vs.add_documents(documents=docs[i:i + BATCH], ids=ids[i:i + BATCH])

final_count = vs._collection.count()
log.info(f"✅ Vectorstore '{COLLECTION}' has {final_count} vectors")
assert final_count == len(chunks), f"Expected {len(chunks)} vectors, got {final_count}"